In [ ]:
# --- STEP 1: Import Necessary Libraries ---
import cv2
import os
import time
import uuid
import numpy as np
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbotmini import Camera, bgr8_to_jpeg
from ipyevents import Event
import traitlets

# --- STEP 2: Directory Setup ---
# Create a folder to store the collected images if it doesn't exist
DATASET_DIR = 'dataset_xy'

try:
    os.makedirs(DATASET_DIR)
    print(f"Directory '{DATASET_DIR}' created successfully.")
except FileExistsError:
    print(f"Directory '{DATASET_DIR}' already exists. New images will be added here.")



In [ ]:
# --- STEP 3: Camera Initialization ---
# Initialize the camera with a resolution of 224x224 (standard for ResNet-18)
# JetPack 4.4.1 handles this via the jetbotmini library
camera = Camera.instance(width=224, height=224)

# --- STEP 4: Define User Interface (Widgets) ---
# image_widget: Displays the live camera feed in the browser
image_widget = widgets.Image(format='jpeg', width=224, height=224)

# count_widget: Shows the total number of images saved so far
count_widget = widgets.IntText(description='Saved Count:', value=len(os.listdir(DATASET_DIR)))

In [15]:
# --- STEP 5: Image Saving Logic ---
def save_snapshot(x, y):
    """
    Saves the current camera frame with coordinate information in the filename.
    """
    # Create a unique filename containing X and Y coordinates
    snapshot_id = 'xy_%03d_%03d_%s' % (x, y, uuid.uuid1())
    filename = os.path.join(DATASET_DIR, snapshot_id + '.jpg')
    
    # Save the current image value from the widget
    with open(filename, 'wb') as f:
        f.write(image_widget.value)
    
    # Update the counter on the screen
    count_widget.value = len(os.listdir(DATASET_DIR))
    print(f"Saved: {filename}") # 확인을 위해 프린트 추가

# --- STEP 6: Event Handling for Mouse Clicks (Upgraded) ---
def on_click(event):
    """
    Triggered when the user clicks on the image_widget.
    Handles different coordinate key names for better compatibility.
    """
    # 1. 클릭 이벤트인지 확인
    if event.get('event') == 'click':
        # 2. 좌표 가져오기 (환경에 따라 이름이 다를 수 있어 둘 다 체크)
        x = event.get('imageX', event.get('canvasX'))
        y = event.get('imageY', event.get('canvasY'))
        
        # 3. 좌표가 정상적으로 들어왔다면 저장
        if x is not None and y is not None:
            save_snapshot(int(x), int(y))
            # 주피터 셀 아래에 로그를 남겨서 작동 여부를 확인합니다.
            print(f"Click detected! -> X: {x}, Y: {y}")

In [ ]:
import cv2
import ipywidgets.widgets as widgets
from jetbotmini import bgr8_to_jpeg
import traitlets

# 1. 좌표 조절용 슬라이더 (0.1 단위 정밀 조절 가능하게 하려면 FloatSlider도 좋지만 여기선 Int로 진행)
target_x_slider = widgets.IntSlider(min=0, max=224, step=1, value=112, description='Steer X')
target_y_slider = widgets.IntSlider(min=0, max=224, step=1, value=150, description='Look-ahead Y')

# 2. 캡처 버튼
capture_btn = widgets.Button(description='Capture Snapshot', button_style='danger', icon='camera')

# 3. 새로운 이미지 처리 함수 (십자선 그리기)
def update_image_with_crosshair(change):
    # 카메라에서 들어온 원본 프레임 (BGR8 포맷)
    frame = change['new']
    
    # 디스플레이용 복사본 생성
    display_frame = frame.copy()
    
    # 현재 슬라이더 값 가져오기
    x, y = target_x_slider.value, target_y_slider.value
    
    # 실시간 십자선 그리기 (초록색, 두께 1)
    cv2.line(display_frame, (x, 0), (x, 224), (0, 255, 0), 1) # 세로선
    cv2.line(display_frame, (0, y), (224, y), (0, 255, 0), 1) # 가로선
    # 중심점에 작은 원 추가
    cv2.circle(display_frame, (x, y), 5, (0, 255, 0), -1)
    
    # 십자선이 그려진 이미지를 위젯에 업데이트
    image_widget.value = bgr8_to_jpeg(display_frame)

# 4. 카메라 데이터 관찰 시작 (기존 dlink 대신 observe 사용)
# camera_link.unlink() # 혹시 기존 링크가 살아있다면 연결 해제 필요
camera.observe(update_image_with_crosshair, names='value')

# 5. 저장 함수 수정 (원본 이미지 저장)
def save_clean_snapshot(b):
    # 십자선이 없는 camera.value(원본)를 저장합니다.
    x, y = target_x_slider.value, target_y_slider.value
    
    snapshot_id = 'xy_%03d_%03d_%s' % (x, y, uuid.uuid1())
    filename = os.path.join(DATASET_DIR, snapshot_id + '.jpg')
    
    # 원본 BGR8 데이터를 JPEG로 변환하여 저장
    with open(filename, 'wb') as f:
        f.write(bgr8_to_jpeg(camera.value))
    
    count_widget.value = len(os.listdir(DATASET_DIR))
    print(f"Captured Clean Image: {filename}")

capture_btn.on_click(save_clean_snapshot)

# 6. UI 배치
ui_controls = widgets.VBox([
    widgets.HBox([target_x_slider, target_y_slider]),
    capture_btn,
    count_widget
])

display(widgets.VBox([image_widget, ui_controls]))

In [ ]:
!zip -r dataset_xy.zip dataset_xy/